# Lecture 9: Recursion

Based on Chapter 9 of *Learning JavaScript Data Structures and Algorithms*.

Recursion solves a problem by solving smaller versions of the same problem. This lecture connects the stack from Chapter 4 to the tree and graph algorithms ahead.

**Learning goals**
- Identify a base case, a recursive case, and progress toward termination.
- Implement and compare iterative and recursive factorial and Fibonacci algorithms.
- Trace stack growth and unwinding; explain stack overflow.
- Use memoization and trampolines for different recursion problems.
- Solve the Power of Two exercise and explain its complexity.

## Running this notebook
Select **JavaScript (Node.js)** and choose **Run All Cells**. These are JavaScript adaptations of the supplied TypeScript: type annotations are omitted so no compiler or extra package is required. Run the definition cells before their demonstrations. Definitions use `var` so cells can be rerun.

The opening dialogue uses predefined answers instead of `readline`, which would wait for terminal input. The stack-overflow demonstration catches its expected error. Large factorials print only their digit count. Look for `All Lecture 9 checks passed.` at the end.

## 1. Understanding recursion
“To understand recursion, one must first understand recursion.” — Author unknown

Imagine searching for a book one library floor at a time. Stop if the book is found or there are no floors left; otherwise search the next floor.

Every recursive solution needs:
1. A **base case** with a directly known result.
2. A **recursive case** that calls the function on a smaller or simpler problem.
3. **Progress** that eventually reaches the base case. Merely having a base case is insufficient if calls move away from it.

Predict how many questions the following answers produce. Then change the array and rerun.

In [1]:
var understandRecursion = function understandRecursion(answers, index = 0) {
  if (index >= answers.length) {
    console.log('No answers left. Pause and review the base case.');
    return false;
  }
  const answer = answers[index];
  console.log('Do you understand recursion? (y/n)', answer);
  if (answer.trim().toLowerCase() === 'y') {
    console.log('Excellent! You have grasped recursion.');
    return true;
  }
  console.log("Let's try again...");
  return understandRecursion(answers, index + 1);
};
understandRecursion(['n', 'no', 'not yet', 'y']);

Do you understand recursion? (y/n) n


Let's try again...


Do you understand recursion? (y/n) no


Let's try again...


Do you understand recursion? (y/n) not yet


Let's try again...


Do you understand recursion? (y/n) y


Excellent! You have grasped recursion.


true

### When recursion fits
Recursion follows the shape of trees, nested directories, and other hierarchical data. It also expresses mathematical definitions such as factorial naturally. Divide-and-conquer algorithms use recursion to solve smaller subproblems and combine results, although not every recursive algorithm is divide-and-conquer.

Choose recursion when its clarity justifies function-call and stack overhead. A loop is often preferable for a long, simple sequence of steps.

The chapter's `readline.question` version repeats through asynchronous callbacks: earlier calls return before later input arrives. The synchronous example above is better for observing stack growth.

## 2. Factorial: iteration and recursion
For a non-negative integer n, **n! = n × (n − 1) × … × 1**, with **0! = 1**.

Recursive definition: `factorial(n) = n * factorial(n - 1)` for n > 1; return 1 for n = 0 or 1. Both implementations below follow the chapter's contract: invalid input returns `undefined`.

**Predict:** What do 0, 1, 5, −1, and 2.5 produce?

In [2]:
var factorialIterative = function factorialIterative(number) {
  if (!Number.isInteger(number) || number < 0) return undefined;
  let total = 1;
  for (let n = number; n > 1; n--) total *= n;
  return total;
};
var factorial = function factorial(number) {
  if (!Number.isInteger(number) || number < 0) return undefined;
  const calculate = (n) => {
    if (n === 0 || n === 1) return 1;
    return n * calculate(n - 1);
  };
  return calculate(number);
};
console.table([0, 1, 5, -1, 2.5].map(n => ({
  n, iterative: factorialIterative(n), recursive: factorial(n)
})));

┌─────────┬─────┬───────────┬───────────┐
│ (index) │ n   │ iterative │ recursive │
├─────────┼─────┼───────────┼───────────┤
│ 0       │ 0   │ 1         │ 1         │
│ 1       │ 1   │ 1         │ 1         │
│ 2       │ 5   │ 120       │ 120       │
│ 3       │ -1  │ undefined │ undefined │
│ 4       │ 2.5 │ undefined │ undefined │
└─────────┴─────┴───────────┴───────────┘


## 3. The JavaScript call stack
The stack follows **Last-In, First-Out (LIFO)**. Each active call retains its local state and any pending work. For factorial, multiplication waits for the smaller factorial to return.

```text
Calls grow                         Results unwind
calculate(5): wait for 4!           5 × 24 = 120
  calculate(4): wait for 3!           4 × 6 = 24
    calculate(3): wait for 2!           3 × 2 = 6
      calculate(2): wait for 1!           2 × 1 = 2
        calculate(1): return 1             1
```

There are five active `calculate` calls at the base case, plus the wrapper and surrounding execution frames. The cell below records entry and return order. Set `showNativeTrace` to `true` to also inspect `console.trace()`; exact frame formatting varies by runtime.

In [3]:
var factorialTraced = function factorialTraced(number, showNativeTrace = false) {
  if (!Number.isInteger(number) || number < 0) return undefined;
  const calculate = (n, depth) => {
    const indent = '  '.repeat(depth);
    console.log(`${indent}call calculate(${n})`);
    if (showNativeTrace) console.trace(`n = ${n}`);
    const result = n <= 1 ? 1 : n * calculate(n - 1, depth + 1);
    console.log(`${indent}return ${result}`);
    return result;
  };
  return calculate(number, 0);
};
console.log('5! =', factorialTraced(5));

call calculate(5)


  call calculate(4)


    call calculate(3)


      call calculate(2)


        call calculate(1)


...

      return 2


    return 6


  return 24


return 120


5! = 120


### Stack overflow
A missing base case repeatedly creates calls until the engine's stack limit is reached. A valid base case can also be too far away for the available stack. The limit depends on the runtime and function shape; it is not a fixed JavaScript constant.

This deliberately broken function catches the resulting error so the lecture can continue. Its count is an observation for this execution, not a portable recursion limit.

In [4]:
(() => {
  let count = 0;
  function recursiveFn() {
    count++;
    recursiveFn();
  }
  try {
    recursiveFn();
  } catch (error) {
    if (!(error instanceof RangeError)) throw error;
    console.log(`Calls observed: ${count}; ${error.name}: ${error.message}`);
  }
})();

Calls observed: 10555; RangeError: Maximum call stack size exceeded


## 4. Tail recursion and exact factorials
A call is in **tail position** when the caller has no work left after it returns. `return n * calculate(n - 1)` is not a tail call: multiplication is pending. An accumulator computes that multiplication before making the next call.

Tail-call optimization can reuse a stack frame where supported. Do not rely on it for this notebook: tail-recursive syntax alone does not guarantee bounded stack usage. The trampoline in the next section makes the execution strategy explicit.

`Number` has finite integer precision and eventually overflows to `Infinity`. `BigInt` stores exact integers, but does not remove stack limits. Use a small input for this direct recursive version.

In [5]:
var factorialTail = function factorialTail(n) {
  if (typeof n !== 'bigint') throw new TypeError('Input must be a bigint');
  if (n < 0n) throw new RangeError('Input must be non-negative');
  const helper = (remaining, accumulator) => {
    if (remaining <= 1n) return accumulator;
    return helper(remaining - 1n, accumulator * remaining);
  };
  return helper(n, 1n);
};
console.log('5! =', factorialTail(5n).toString());
console.log('25! (exact) =', factorialTail(25n).toString());

5! = 120


25! (exact) = 15511210043330985984000000


## 5. The trampoline pattern
Return a **thunk** (a function representing the next step) instead of immediately calling the next recursive step. A loop executes each thunk after the previous call has returned, so stack depth stays bounded.

The chapter's TypeScript type is `type Bounce<T> = T | (() => Bounce<T>)`. Here the final value is a `BigInt`; this simple trampoline assumes a function result always represents another step.

Trampolines avoid growing the call stack but allocate wrapper functions. Their speed depends on the workload; a plain loop is often simpler for factorial. Exact large results still require memory and increasingly expensive arithmetic.

In [6]:
var trampoline = function trampoline(fn) {
  return (...args) => {
    let result = fn(...args);
    while (typeof result === 'function') result = result();
    return result;
  };
};
var factorialTrampoline = function factorialTrampoline(n, accumulator = 1n) {
  if (typeof n !== 'bigint') throw new TypeError('Input must be a bigint');
  if (n < 0n) throw new RangeError('Input must be non-negative');
  if (n <= 1n) return accumulator;
  return () => factorialTrampoline(n - 1n, accumulator * n);
};
var factorialBounced = trampoline(factorialTrampoline);
console.log('5! =', factorialBounced(5n).toString());
console.log('Digits in 20000!:', factorialBounced(20000n).toString().length);

5! = 120


Digits in 20000!: 77338


## 6. The Fibonacci sequence
Starting at index zero: **0, 1, 1, 2, 3, 5, 8, 13, 21, 34, …**

`F(0) = 0`, `F(1) = 1`, and `F(n) = F(n - 1) + F(n - 2)`.

### Iterative Fibonacci
Keep the previous two values and advance them one position at a time. Invalid inputs throw a `RangeError`.

In [7]:
var validateFibonacciInput = function validateFibonacciInput(n) {
  if (!Number.isInteger(n) || n < 0) {
    throw new RangeError('Input must be a non-negative integer');
  }
};
var fibonacciIterative = function fibonacciIterative(n) {
  validateFibonacciInput(n);
  if (n < 2) return n;
  let prevPrev = 0;
  let prev = 1;
  for (let i = 2; i <= n; i++) {
    const current = prev + prevPrev;
    prevPrev = prev;
    prev = current;
  }
  return prev;
};
console.log(Array.from({ length: 10 }, (_, n) => fibonacciIterative(n)));

[
  0, 1,  1,  2,  3,
  5, 8, 13, 21, 34
]


### Naive recursive Fibonacci
The code mirrors the mathematical definition but repeats work. In `F(5)`, `F(3)` appears twice:

```text
F(5)
├─ F(4)
│  ├─ F(3)
│  └─ F(2)
└─ F(3)
   ├─ F(2)
   └─ F(1)
```

Only part of the call tree is shown. Keep demonstration inputs small; the total number of calls grows exponentially.

In [8]:
var fibonacci = function fibonacci(n) {
  validateFibonacciInput(n);
  const calculate = (value) => {
    if (value < 2) return value;
    return calculate(value - 1) + calculate(value - 2);
  };
  return calculate(n);
};
console.log('F(5) =', fibonacci(5));
console.log('F(10) =', fibonacci(10));

F(5) = 5


F(10) = 55


### Fibonacci with memoization
Cache each result and reuse it when the same subproblem appears again. Check `cached !== undefined`, rather than truthiness: zero is a valid cached result.

The cache below is created once per public invocation and shared by its recursive helper through a closure. It removes repeated calculations but still has a linear-depth call stack.

In [9]:
var fibonacciMemoization = function fibonacciMemoization(n) {
  validateFibonacciInput(n);
  const memo = [0, 1];
  const calculate = (value) => {
    const cached = memo[value];
    if (cached !== undefined) return cached;
    memo[value] = calculate(value - 1) + calculate(value - 2);
    return memo[value];
  };
  return calculate(n);
};
console.table([0, 1, 5, 10, 20].map(n => ({
  n, iterative: fibonacciIterative(n),
  recursive: fibonacci(n), memoized: fibonacciMemoization(n)
})));
console.log('F(40), memoized:', fibonacciMemoization(40));

┌─────────┬────┬───────────┬───────────┬──────────┐
│ (index) │ n  │ iterative │ recursive │ memoized │
├─────────┼────┼───────────┼───────────┼──────────┤
│ 0       │ 0  │ 0         │ 0         │ 0        │
│ 1       │ 1  │ 1         │ 1         │ 1        │
│ 2       │ 5  │ 5         │ 5         │ 5        │
│ 3       │ 10 │ 55        │ 55        │ 55       │
│ 4       │ 20 │ 6765      │ 6765      │ 6765     │
└─────────┴────┴───────────┴───────────┴──────────┘


F(40), memoized: 102334155


### Measure repeated work
Counting helper invocations makes the difference reproducible without depending on machine timing. A cached lookup still counts as an invocation; a computed subproblem counts only when it was not in the cache. Predict the difference for n = 20.

In [10]:
(() => {
  function measure(n, useMemo) {
    let calls = 0;
    let computed = 0;
    const memo = [0, 1];
    function calculate(value) {
      calls++;
      if (useMemo && memo[value] !== undefined) return memo[value];
      if (value < 2) return value;
      computed++;
      const result = calculate(value - 1) + calculate(value - 2);
      if (useMemo) memo[value] = result;
      return result;
    }
    const result = calculate(n);
    return { strategy: useMemo ? 'memoized' : 'naive', n, result, calls, computed };
  }
  console.table([measure(10, false), measure(10, true),
                 measure(20, false), measure(20, true)]);
})();

┌─────────┬────────────┬────┬────────┬───────┬──────────┐
│ (index) │ strategy   │ n  │ result │ calls │ computed │
├─────────┼────────────┼────┼────────┼───────┼──────────┤
│ 0       │ 'naive'    │ 10 │ 55     │ 177   │ 88       │
│ 1       │ 'memoized' │ 10 │ 55     │ 19    │ 9        │
│ 2       │ 'naive'    │ 20 │ 6765   │ 21891 │ 10945    │
│ 3       │ 'memoized' │ 20 │ 6765   │ 39    │ 19       │
└─────────┴────────────┴────┴────────┴───────┴──────────┘


## 7. Efficiency and trade-offs
Assume constant-time arithmetic for these introductory comparisons. Space means peak auxiliary space, including the call stack.

| Algorithm | Time | Extra space | Reason |
| --- | --- | --- | --- |
| Iterative factorial | O(n) | O(1) | One loop and a running product |
| Recursive factorial | O(n) | O(n) | Linear chain of pending calls |
| Tail-recursive factorial without optimization | O(n) | O(n) | Tail position alone does not remove frames |
| Trampolined factorial | O(n) steps | O(1) live control state | One thunk runs at a time |
| Iterative Fibonacci | O(n) | O(1) | Keep two previous values |
| Naive recursive Fibonacci | O(2ⁿ) upper bound | O(n) | Exponentially many calls, linear maximum depth |
| Memoized Fibonacci | O(n) | O(n) | Cache plus recursive stack |

The tighter bound for naive Fibonacci is Θ(φⁿ), where φ ≈ 1.618. BigInt multiplication is not constant-time: exact factorials require growing values, so the trampoline's O(1) control-state claim excludes result storage and arithmetic workspace. Number-based Fibonacci also loses exact integer precision for sufficiently large n (the sequence exceeds `Number.MAX_SAFE_INTEGER` at F(79)).

**Discuss:** Why does memoization reduce Fibonacci's time but not its maximum recursion depth? Why does a trampoline solve a different problem?

## 8. Worked exercise: Power of Two
Given an integer n, return whether it is a power of two.

- Invalid, zero, or negative input → false.
- n = 1 → true, because 2⁰ = 1. Check this before rejecting odd numbers.
- Other odd numbers → false.
- Otherwise, solve the smaller problem n / 2.

Trace 16 → 8 → 4 → 2 → 1 before running the solution. This has four recursive transitions and five function invocations. The worst-case time and stack space are O(log n) for positive n in the usual input-size model.

In [11]:
var isPowerOfTwo = function isPowerOfTwo(n) {
  if (!Number.isInteger(n) || n <= 0) return false;
  if (n === 1) return true;
  if (n % 2 !== 0) return false;
  return isPowerOfTwo(n / 2);
};
console.table([1, 16, 18, 0, -2, 2.5].map(n => ({ n, result: isPowerOfTwo(n) })));

┌─────────┬─────┬────────┐
│ (index) │ n   │ result │
├─────────┼─────┼────────┤
│ 0       │ 1   │ true   │
│ 1       │ 16  │ true   │
│ 2       │ 18  │ false  │
│ 3       │ 0   │ false  │
│ 4       │ -2  │ false  │
│ 5       │ 2.5 │ false  │
└─────────┴─────┴────────┘


## 9. Practice before checking solutions
1. Write `sumTo(n)` recursively for non-negative integers. Identify its base case and decreasing input; expect `sumTo(5) === 15` and `sumTo(0) === 0`.
2. Convert Power of Two to a loop. What changes in its extra space usage?
3. Trace `factorial(4)` by hand, showing both calls and returns.
4. Explain why a memo cache must be shared among recursive calls rather than recreated in every helper call.
5. Change the dialogue answers to `[]`, `['n']`, and `['Y']`. Explain how each run terminates.

The next cell contains sample solutions for exercises 1 and 2. Try your own first.

In [12]:
var sumTo = function sumTo(n) {
  validateFibonacciInput(n); // Same non-negative integer contract.
  if (n === 0) return 0;
  return n + sumTo(n - 1);
};
var isPowerOfTwoIterative = function isPowerOfTwoIterative(n) {
  if (!Number.isInteger(n) || n <= 0) return false;
  while (n > 1) {
    if (n % 2 !== 0) return false;
    n /= 2;
  }
  return true;
};
console.log('sumTo(5):', sumTo(5));
console.log('Iterative power of two:', isPowerOfTwoIterative(16));
// The loop uses O(log n) time and O(1) extra space.

sumTo(5): 15


Iterative power of two: true


## 10. Runnable checks
These assertions verify base cases, invalid inputs, agreement across implementations, exact BigInt results, a deep trampoline run, and the exercise solutions. A failed check throws an error.

In [13]:
(() => {
  const assert = require('node:assert/strict');
  for (const fn of [factorial, factorialIterative]) {
    assert.equal(fn(0), 1);
    assert.equal(fn(1), 1);
    assert.equal(fn(5), 120);
    for (const invalid of [-1, 2.5, NaN, Infinity, '5']) {
      assert.equal(fn(invalid), undefined);
    }
  }
  for (let n = 0; n <= 12; n++) {
    assert.equal(factorial(n), factorialIterative(n));
    assert.equal(factorialBounced(BigInt(n)), factorialTail(BigInt(n)));
  }
  assert.equal(factorialBounced(0n), 1n);
  assert.equal(factorialTail(25n), 15511210043330985984000000n);
  assert.throws(() => factorialTail(-1n), RangeError);
  assert.throws(() => factorialBounced(-1n), RangeError);
  assert.throws(() => factorialBounced(5), TypeError);
  assert.equal(factorialBounced(20000n).toString().length, 77338);
  const expected = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55];
  for (const fn of [fibonacciIterative, fibonacci, fibonacciMemoization]) {
    expected.forEach((value, n) => assert.equal(fn(n), value));
    for (const invalid of [-1, 1.5, NaN, Infinity, '5']) {
      assert.throws(() => fn(invalid), RangeError);
    }
  }
  assert.equal(fibonacciMemoization(40), 102334155);
  for (const fn of [isPowerOfTwo, isPowerOfTwoIterative]) {
    for (const n of [1, 2, 4, 16, 1024]) assert.equal(fn(n), true);
    for (const n of [0, -2, 3, 18, 2.5, NaN, Infinity]) assert.equal(fn(n), false);
  }
  assert.equal(sumTo(0), 0);
  assert.equal(sumTo(5), 15);
  console.log('All Lecture 9 checks passed.');
})();

All Lecture 9 checks passed.


## Lecture review
Recursion requires a stopping condition and progress toward it. The call stack explains pending work, return order, and depth limits. Memoization avoids repeating subproblems; trampolines avoid accumulating recursive stack frames. Iteration often uses less auxiliary memory for linear computations.

**Exit questions:** What is factorial's pending work? Why is naive Fibonacci expensive? Does tail-recursive syntax guarantee constant stack space? Which approach would you choose for a deeply nested structure?

Next: trees, where recursion follows each node's child subtrees.